In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import numpy as np
from time import sleep

from URBasic import Joint6D
from URBasic import TCP6D
from URBasic import TCP6DDescriptor

import src.calibration as cal
import src.space as sp

SIMULATION = True
# Create a new ISCoin object
# UR3e1 IP (closest to window): 10.30.5.158
# UR3e2 IP: 10.30.5.159
robot_ip = "10.30.5.159"
robot_opened_gripper_size_mm = 40. # Max oppenning of gripper

# Base robot position in Joint space
j_o = Joint6D.createFromRadians(
    -1.5708, -1.5708, -1.0472, -2.0944, 1.5708, 0.0
)

In [ ]:
# Simulator (swap just this line)
if SIMULATION:
    from my_simulation import ISCoinSim as ISCoin
    robot_ip = ""

    iscoin = ISCoin(opened_gripper_size_mm=robot_opened_gripper_size_mm)
else:
    from URBasic import ISCoin

    iscoin = ISCoin(robot_ip, robot_opened_gripper_size_mm)

    if not iscoin.isInitOk:
        print("Error robot not initialised")
        sys.exit(1)

    iscoin.robot_control.reset_error()

In [ ]:
robot_arm = iscoin.robot_control

In [ ]:
if not SIMULATION:
    tcps = cal.collect_data(robot_arm, 20)
else:
    tcps = [
        [0.0024946337740892888, -0.3594913915339868, 0.06033167700558813, -0.62069755529755, -2.369727486212462, -0.13752393611863362],
        [-0.07482225018547103, -0.35685269130655106, 0.07009337653443365, 1.9372057315922462, 2.190232377619723, 0.6719840297817087],
        [-0.08776793211801745, -0.2979812165503483, 0.051154992555969794, 2.3781388566898687, 0.2623998383341909, 0.7178051284770465],
        [-0.0006933205535983589, -0.340924257935631, 0.06551204074775754, -2.879088789274111, -0.4716877213448937, 0.8276834472055694],
        [-0.04094004360899719, -0.3437262194580261, 0.0786140415986546, -3.087426800813378, 0.2627406174702693, 0.06421242457662117],
        [0.026621421400990636, -0.3469050147213359, 0.033956680032955974, -2.5014091646201537, -0.22918589994059693, 1.5168173293250897],
    ]

In [ ]:
tcp_offset = cal.get_tcp_offset(tcps)
if not SIMULATION:
    robot_arm.set_tcp(tcp_offset)
    robot_arm.freedrive_mode()
    tcp_ref = None
    while not tcp_ref:
        i = input(f"Confirm reference position for TCP validation. y/n")
        if i.lower()[0] == "y":
            robot_arm.end_freedrive_mode()
            joint_ref = robot_arm.get_actual_joint_positions()

    cal.validate_calibration(robot_arm, joint_ref)

In [ ]:
if not SIMULATION:
    p_world = np.array([
        #TODO:  Give the measure points of the object
        #       Here the plaque.png
        [-5.5, -19.5, -2],
        [-5.5, 105.5, -2],
        [69.5, 105.5, -2],
        [69.5, -19.5, -2],
    ])/1000
    p_tcp = sp.collect_data(robot_arm, p_world)
else:
    p_world = np.array([
        [-5.5, -19.5, -2],
        [-5.5, 105.5, -2],
        [69.5, 105.5, -2],
        [69.5, -19.5, -2],
    ])/1000

    p_tcp = np.array([
        [-0.05, -0.2, 0.2500, 0, 3.14, 0],
        [-0.05, -0.4, 0.2500, 0, 3.14, 0],
        [ 0.05, -0.4, 0.2500, 0, 3.14, 0],
        [ 0.05, -0.2, 0.2500, 0, 3.14, 0],
    ])

In [ ]:
world_to_tcp = sp.affine_transform_3D(p_world, p_tcp)

In [ ]:
import src.draw as draw

center      = (0.032, 0.046, 0)
rx, ry, rz  =  0,     3.14,  0
path = draw.draw_cercle(center, 0.03, 32)

In [ ]:
motion = []
for p in path:
    x,y,z = world_to_tcp(p)
    t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
    m = TCP6DDescriptor(t, v=1)
    motion.append(m)

robot_arm.movej(j_o)
robot_arm.movel_waypoints(motion)
robot_arm.movej(j_o)